# Classification of Wikipedia Articles
Wikipedia is an encyclopedia that covers a large amount of diverse topics. All articles are created, corrected and updated by individuals. The goal is to correctly document as many topics as possible by collecting the knowledge of a large number of people. However, some articles stand out due to their completeness, scope and presentation, and for this they are marked with the distinction of the Excellent Article. 

As part of the Natural Language Processing lecture, a classification of Wikipedia articles is to be carried out as a sub-task of an assignment with the goal of being able to identify excellent articles. This notebook contains the code to accomplish this goal and is structured as follows:

- [1. Imports](article_classification.ipynb#1-imports)
- [2. Automated Data Check](article_classification.ipynb#2-check-data-availability)
- [3. Load and Process the Data](article_classification.ipynb#3-data-processing)
- [4. Train Neural Network]()
	- [4.1 Train/Validation/Test-Split](#thema1)
	- [4.2 Pre-Process the Texts](#thema2)
	- [4.3 Create Neural Network](#thema3)
	- [4.4 Train the Neural Network]()
	- [4.5 Validation of Results]()
- [5. Conclusion](#schluss)


## 1. Imports
Import the requiered libraties into the notebook.
If some libraries are not installed, you can use the `requierements.txt` and run
```
$ pip install -r requirements.txt
```
in the terminal.

In [1]:
# Automated Data Download & Extraction
import os
import subprocess

# Effecient Processing of Wikipedia Dump
import mwxml # Import mwxml for effecient iteration of dump
import mwparserfromhell # Extract Features from Text

# Data Storage
import re # Import RegEx
from tqdm.notebook import tqdm # Import Progress-Bar
import pandas as pd
import numpy as np

# Pre-Processing
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Neural Network
import tensorflow as tf
from tensorflow.keras.layers import Embedding, Conv1D, LSTM, Dense
from tensorflow.keras.layers import BatchNormalization, Dropout, MaxPool1D
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.regularizers import L1L2

# Evaluation
import plotly.graph_objects as go
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
from imblearn.metrics import geometric_mean_score

## 2. Check Data Availability
In order to get the training data, a backup of the current wikipedia encyclopedie is needed. <br>
These dumps can be downloaded in every language by changing the url to https://dumps.wikimedia.org/[`insert_language (e.g. de, en)`]wiki/latest/. <br>
To make this example easy to run and have an equal data foundation, the download was automated with this cell. If the correct file already exists in the data folder, the download will be skipped. <br>
The file will be donwloaded as an `.bz2`-Archive and must be extracted before use. <br>
<b>Note: Due to the size of the file, the download may take a longer time depending on your internet connection.</b>

In [ ]:
if(not(os.path.exists("../Data"))):
   os.makedirs('../Data')

if(not(os.path.exists("../Data/dewiki-latest-pages-articles-multistream.xml"))):
    print("Articles-File not found. Download started... (this might take a while):")
    subprocess.call(['sh', '../bin/download-and-unzip-data.sh'])
print("Articles-File available at: ../Data/dewiki-latest-pages-articles-multistream.xml")

## 3. Data Processing
A big challenge is the effecient processing of the wikipedia dump files. Due to its size (ca. 26,8 GB), it is not possible to load the whole file into the memory. There are different approaches to deal with this problem but we chose to use the `mwxml`-library, which creates an generator-object that returns a single Wikipedia article at a time. In order to use natural language processing for the classification of the individual articles, the respecitve texts must be extracted. However, this poses another challenge due to the HTML-formatting.

In [ ]:
def dump_to_dataset(path_to_dump:str, n_samples:int=-1, balance_ratio:float=-1.0, random_state:int=456) -> pd.DataFrame:
    """
    Creates a dataset out of a wikipedia dump that contains the text and label of each article.

    If the argument 'n_samples' is passed, only the specified number of samples will be processed.

    If the argument 'balance_ratio' is passed, the dataset will be resampled in order to achieve the desired ratio.

    Parameters
    ----------
    path_to_dump : str. Relative file path to the unzipped wikipedia dump
    n_samples: int, optional.  Samples to be processed, if not set the whole dataset will be iterated
    balance_ratio: float, optional. Desired resampling ratio of majority and minority class, if not set no resampling will be performed
    random_state: int, optional. Random state for resampling, if not set default random state will be used

    Returns
    -------
    df: pd.DataFrame. Pandas DataFrame that contains the processed wikipedia articles with labels
    """
    data_index = [] # Empty array for the article ids
    label_index = [] # Empty array for the labels
    dump = mwxml.Dump.from_file(open(path_to_dump)) # Load Wikipedia dump and create generator object
    i = 0 # Set iteration variable to zero
    print("Step 1: Indexing Wikipedia Dataset")
    pbar = tqdm(total= (n_samples if (n_samples>1) else 5425758)) # Create statusbar
    for page in dump: # Iterate over pages in dump
        for revisions in page: # Iterate over revisions of page
            try:
                if((revisions.page.namespace == 0) & (not revisions.page.redirect) & ("Liste" not in revisions.page.title)):
                    if(re.search(r"{{Exzellent[|](\d*).(\D*)(\d*)[|](\d*)}}", revisions.text)): # If article is marked as excelennt
                        label_index.append(1) # Add 1 (positive) as label to array
                    else:
                        label_index.append(0) # Add 0 (negative) as label to array
                    data_index.append(revisions.page.id) # Add page id to array
                    i += 1 # Increment iteration variable
                pbar.update(1) # Updata status bar
            except Exception as e:
                print(e)
        if(i>=(n_samples if (n_samples>1) else 5425758)): # If n_samples or end of dump is reached
            break # End iterations
    print(len(data_index))
    pbar.close() # Stop progress bar
    print("Step 2: Resample Collected Data")
    temp_df = pd.DataFrame({'text_id': data_index, 'label': label_index}, columns=['text_id', 'label']) #  Create temporal dataframe
    minority_class = temp_df[temp_df['label'] == 1]
    downsampled_majority = resample(
        temp_df[temp_df['label'] == 0], # Get majority class samples
        replace=False, # Undersampling
        n_samples=int(balance_ratio * len(minority_class)), # Set resampling ratio
        random_state=random_state # Add random state
    )
    downsampled_df = pd.concat([downsampled_majority, minority_class]).sample(frac=1) # Combine downsampled majority class with minority class
    valid_list = downsampled_df["text_id"].values # Create list with valid article ids
    print("Step 3: Create Balanced Dataset")
    text = np.memmap('../Data/article_text.dat', dtype='object', mode='w+', shape=(len(valid_list), 1)) # Create memmap for texts
    label = np.memmap('../Data/label.dat', dtype=np.int8, mode='w+', shape=(len(valid_list), 1)) # Create memmap for labels
    dump = mwxml.Dump.from_file(open(path_to_dump)) # Load Wikipedia dump and create generator object
    i = 0 # Reset iteration variable to zero
    pbar2 = tqdm(total=len(valid_list)) # Create statusbar
    for page in dump: # Iterate over pages in dump
        for revisions in page: # Iterate over revisions in page
            try:
                if("Liste" not in revisions.page.title): # Exclude non-article revisions
                    if revisions.page.id in valid_list: # If article is in resampled scope
                        temp_text = mwparserfromhell.parse(revisions.text) # Parse text
                        if any((re.match(r"{{Exzellent[|](\d*).(\D*)(\d*)[|](\d*)}}", str(template)) for template in temp_text.filter_templates())): # If article is markes as excellent
                            label[i] = 1 # Add 1 (positive) as label to memmap
                        else:
                            label[i] = 0 # Add 0 (negative) as label to memmap
                        text[i] = re.sub(r"Exzellent (\d*).(\D*).(\d*)", "", (" ".join((" ".join(list(map(str, temp_text.filter_text())))).split()))) # Add cleaned text to memmap
                        i += 1 # Increment iteration variable
                        pbar2.update(1) # Update status bar
            except Exception as e:
                print(e)
        if(i>=len(valid_list)): # If valid list is completed
            break # Break iteration
    pbar2.close() # Close status bar
    df = pd.DataFrame({'text': np.array(text).ravel(), 'label': list(np.array(label))}, columns=['text', 'label']) # Combine memmap arrays to dataframe
    return df # return dataframe

In [ ]:
path_to_dump = "../Data/dewiki-latest-pages-articles-multistream.xml"
dataframe = dump_to_dataset(path_to_dump=path_to_dump, balance_ratio=1.5)

In [ ]:
dataframe.to_pickle("../Data/processed_dataset.pkl")

## 4. Train Neural Network

In [2]:
dataframe = pd.read_pickle("../Data/processed_dataset.pkl")

### 4.1 Tokenize Text of Articles & Pad Sequences

In [3]:
X = np.array(dataframe["text"].values)
y = np.asanyarray(dataframe["label"].values).astype(np.int16)

In [4]:
np.unique(y, return_counts=True)

(array([0, 1], dtype=int16), array([4193, 2794]))

In [5]:
tokenizer = Tokenizer(
    num_words=10000,
    filters='!"„“#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n'
)
tokenizer.fit_on_texts(X)

In [7]:
X_token = pad_sequences(tokenizer.texts_to_sequences(X), maxlen=10000)

## 4.2 Split Dataset

In [8]:
X_train, X_rest, y_train, y_rest = train_test_split(
    X_token, 
    y,
    stratify=y, 
    test_size=0.3,
    random_state=456
)

X_test, X_val, y_test, y_val = train_test_split(
    X_rest,
    y_rest,
    stratify=y_rest,
    test_size=0.5,
    random_state=456
)

In [9]:
print(len(X_train), len(X_test), len(X_val))

4890 1048 1049


In [10]:
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

Num GPUs Available:  1


### 4.3 Build and compile Neural Network

In [11]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=10000, output_dim=64),
    tf.keras.layers.Conv1D(filters=32, kernel_size=3, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling1D(pool_size=2),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.LSTM(64, kernel_regularizer=L1L2(0, 0.001)),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

opt = tf.keras.optimizers.legacy.Adam(learning_rate=0.001)
model.compile(
    loss='binary_crossentropy', 
    optimizer=opt, 
    metrics=[
        'binary_accuracy'
    ]
)

Metal device set to: Apple M1 Pro

systemMemory: 16.00 GB
maxCacheSize: 5.33 GB



### 4.4 Train Neural Network

In [12]:
earlystopper = EarlyStopping(patience=25, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=0.000001, verbose=1, cooldown=5)

history = model.fit(
    X_train, 
    y_train,
    validation_data=(X_val, y_val),
    epochs=30, 
    batch_size=100,
    verbose=1,
    shuffle=True,
    callbacks=[earlystopper, reduce_lr]
)

Epoch 1/30


2023-06-22 18:15:17.037360: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


49/49 [==============================] - 313s 6s/step - loss: 0.6176 - binary_accuracy: 0.7022 - val_loss: 0.6664 - val_binary_accuracy: 0.6654 - lr: 0.0010
Epoch 2/30
49/49 [==============================] - 290s 6s/step - loss: 0.3045 - binary_accuracy: 0.8988 - val_loss: 1.9177 - val_binary_accuracy: 0.3994 - lr: 0.0010
Epoch 3/30
49/49 [==============================] - 290s 6s/step - loss: 0.2141 - binary_accuracy: 0.9397 - val_loss: 0.7059 - val_binary_accuracy: 0.6254 - lr: 0.0010
Epoch 4/30
49/49 [==============================] - 142s 3s/step - loss: 0.1442 - binary_accuracy: 0.9658 - val_loss: 0.4950 - val_binary_accuracy: 0.7741 - lr: 0.0010
Epoch 5/30
49/49 [==============================] - 133s 3s/step - loss: 0.1074 - binary_accuracy: 0.9785 - val_loss: 0.3962 - val_binary_accuracy: 0.8284 - lr: 0.0010
Epoch 6/30
49/49 [==============================] - 142s 3s/step - loss: 0.0988 - binary_accuracy: 0.9818 - val_loss: 0.3626 - val_binary_accuracy: 0.9123 - lr: 0.0010
Epo

In [13]:
fig = go.Figure(
    data = [
        go.Scatter(y=history.history['loss'], name="train"),
        go.Scatter(y=history.history['val_loss'], name="val"),
    ],
    layout = {"yaxis": {"title": "Loss [BCE]"}, "xaxis": {"title": "Epoch"}, "title": "Model Loss over Epochs"}
)
fig.show()


In [15]:
y_test_predictions = (np.array(model.predict(X_test)) >= 0.5).astype(int)
f1score = f1_score(y_test, y_test_predictions)
gm = geometric_mean_score(y_test, y_test_predictions, average="binary")
auc = roc_auc_score(y_test, y_test_predictions, average="weighted")
precision = precision_score(y_test, y_test_predictions)
recall = recall_score(y_test, y_test_predictions)

33/33 [==============================] - 3s 104ms/step


In [16]:
print(f1score, gm, auc, precision, recall)

0.8983451536643027 0.9160537489968281 0.9160997302229928 0.8899297423887588 0.9069212410501193
